# Task 6: Feature Selection and Recommendation Quality Evaluation

## Objective
So far my recommender only used TF-IDF and cosine similarity on text. In this notebook I check whether the features I feed into the model actually help it recommend meaningful TED Talks, and I choose the best combination for the project.

## How it works (in simple words)
TF-IDF turns each talk's text into numbers, giving more weight to words that are special to that talk and less to words that appear everywhere. Cosine similarity then measures how close two talks are. The talks with the closest numbers become the recommendations.

## The four experiments
| Experiment | Features used |
|---|---|
| Experiment 1 | Title only |
| Experiment 2 | Title + Tags |
| Experiment 3 | Title + Description + Tags |
| Experiment 4 | Title + Description + Transcript + Tags |

In this dataset, "tags" are stored in the `topics` column.

## Important notes
- Only the features change between experiments. Cleaning, TF-IDF settings and the recommendation function stay the same, so the comparison is fair.
- This notebook uses only the cleaned TED Talks dataset. **No simulated data** is used here.


In [1]:
import re
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

# so the tables don't cut off long titles in my screenshots
pd.set_option("display.max_colwidth", 70)
pd.set_option("display.width", 250)

In [2]:
df = pd.read_csv("../data/processed/ted_talks_cleaned.csv")
print(df.shape)
print(list(df.columns))

(4005, 23)
['talk_id', 'title', 'speaker_1', 'all_speakers', 'occupations', 'about_speakers', 'views', 'recorded_date', 'published_date', 'event', 'native_lang', 'available_lang', 'comments', 'duration', 'topics', 'related_talks', 'url', 'description', 'transcript', 'title_clean', 'topics_clean', 'description_clean', 'combined_text']


In [3]:
# column names in one place, so I only change them here if needed
TITLE_COL = "title"
DESC_COL = "description"
TAGS_COL = "topics"            # the dataset calls tags "topics"
TRANSCRIPT_COL = "transcript"

In [4]:
def tidy(text):
    # lowercase, drop punctuation (keeping apostrophes like before), squash extra spaces
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9' ]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()

# empty string for missing values, same decision I took in Task 3
for col in [TITLE_COL, DESC_COL, TAGS_COL, TRANSCRIPT_COL]:
    df[col + "_t"] = df[col].fillna("").map(tidy)

title_text = df[TITLE_COL + "_t"]
desc_text = df[DESC_COL + "_t"]
tags_text = df[TAGS_COL + "_t"]
transcript_text = df[TRANSCRIPT_COL + "_t"]

# each experiment is just a different recipe of the same ingredients
experiments = {
    "Exp 1: Title": title_text,
    "Exp 2: Title + Tags": title_text + " " + tags_text,
    "Exp 3: Title + Desc + Tags": title_text + " " + desc_text + " " + tags_text,
    "Exp 4: Title + Desc + Transcript + Tags": title_text + " " + desc_text + " " + transcript_text + " " + tags_text,
}
names = list(experiments)

In [5]:
vectors = {}
for name, text in experiments.items():
    vectorizer = TfidfVectorizer(stop_words="english")   # stop words handled here, as agreed with mam
    vectors[name] = vectorizer.fit_transform(text)
    print(name, "->", vectors[name].shape)

Exp 1: Title -> (4005, 4848)
Exp 2: Title + Tags -> (4005, 4934)
Exp 3: Title + Desc + Tags -> (4005, 21491)
Exp 4: Title + Desc + Transcript + Tags -> (4005, 70897)


In [6]:
def recommend_talks(title, experiment, top_n=5):
    matches = df.index[df[TITLE_COL].str.lower() == title.lower()]
    if len(matches) == 0:
        print(f"'{title}' not found. Check the spelling.")
        return pd.DataFrame()

    idx = matches[0]
    matrix = vectors[experiment]
    scores = linear_kernel(matrix[idx], matrix).ravel()   # similarity with every talk
    scores[idx] = -1                                      # a talk should never recommend itself
    top = scores.argsort()[::-1][:top_n]

    return pd.DataFrame({
        "rank": range(1, len(top) + 1),
        "title": df.loc[top, TITLE_COL].values,
        "score": scores[top].round(3),
    })

## Choosing the five talks
I searched the titles by keyword and picked five talks from very different subjects, so the differences between the four experiments are easy to see.

In [8]:
def find_talks(keyword, n=8):
    hits = df[df[TITLE_COL].str.contains(keyword, case=False, na=False)]
    return hits[[TITLE_COL]].head(n)

for word in ["climate", "brain", "school", "cancer", "love"]:
    print("\n--- keyword:", word, "---")
    display(find_talks(word))


--- keyword: climate ---


,title
0,Averting the climate crisis
50,Global priorities bigger than climate change
156,A critical look at geoengineering against climate change
209,New thinking on the climate crisis
538,The science behind a climate headline
727,Discovering ancient climates in oceans and ice
1116,Why I must speak out about climate change
1295,Let's prepare for our new climate



--- keyword: brain ---


,title
104,How brain science will change computing
150,3 clues to understanding your brain
203,A look inside the brain in real time
252,The brain in love
255,Brain magic
264,A computer that works like the brain
428,Growing evidence of brain plasticity
463,How cults rewire the brain



--- keyword: school ---


,title
4,Do schools kill creativity?
7,Let's teach religion -- all religion -- in schools
199,My wish: Once Upon a School
286,What's wrong with school lunches
662,Inside a school for suicide bombers
775,My green school dream
841,How state budgets are breaking US schools
978,A short intro to the Studio School



--- keyword: cancer ---


,title
30,Meet the future of cancer research
589,A new strategy in the war on cancer
655,Can we eat to starve cancer?
679,Fighting cancer with dance
836,Understanding cancer through proteomics
975,Fighting a contagious cancer
999,Open-source cancer research
1079,Treating cancer with electric fields



--- keyword: love ---


,title
22,"Why we love, why we cheat"
252,The brain in love
308,"Aliens, love -- where are they?"
610,How I fell in love with a fish
649,"""Love Is a Loaded Pistol"""
741,How Mr. Condom made Thailand a better place for life and love
752,"Poverty, money -- and love"
1328,Love letters to strangers


## Choosing the five talks
I searched the titles by keyword and picked five talks from very different subjects, so the differences between the four experiments are easy to see.

| Subject | Selected talk |
|---|---|
| Climate | Averting the climate crisis |
| Brain science | Growing evidence of brain plasticity |
| Education | Do schools kill creativity? |
| Health | Can we eat to starve cancer? |
| Relationships | Why we love, why we cheat |

## What the output below shows
1. **Side-by-side tables:** the top 5 recommendations from each experiment, for each selected talk.
2. **Overlap table:** how many of the 5 recommendations two experiments share (5 means identical, 0 means completely different). It shows how strongly the extra features change the results.
3. **Weak matches in title-only:** a similarity score of 0 means nothing in the titles matched, so that recommendation is basically random.

In [9]:
selected_titles = [
    "Averting the climate crisis",
    "Growing evidence of brain plasticity",
    "Do schools kill creativity?",
    "Can we eat to starve cancer?",
    "Why we love, why we cheat",
]

# safety check: stop here if any title is misspelled
missing = [t for t in selected_titles if t.lower() not in df[TITLE_COL].str.lower().values]
if missing:
    print("These titles were not found, copy them again from the list above:", missing)
else:
    # 1) side-by-side tables
    for talk in selected_titles:
        table = pd.DataFrame({"Rank": [1, 2, 3, 4, 5]})
        for name in names:
            table[name] = recommend_talks(talk, name)["title"].values
        print("\nSelected talk:", talk)
        display(table.set_index("Rank"))

    # 2) how much do the recommendations change?
    rows = []
    for talk in selected_titles:
        top = {n: set(recommend_talks(talk, n)["title"]) for n in names}
        rows.append({
            "Talk": talk[:40],
            "Exp1 vs Exp2": len(top[names[0]] & top[names[1]]),
            "Exp1 vs Exp3": len(top[names[0]] & top[names[2]]),
            "Exp1 vs Exp4": len(top[names[0]] & top[names[3]]),
            "Exp3 vs Exp4": len(top[names[2]] & top[names[3]]),
        })
    print("\nOverlap between experiments (out of 5):")
    display(pd.DataFrame(rows))

    # 3) weak matches in title-only
    print("\nExperiment 1 recommendations with zero similarity:")
    for talk in selected_titles:
        scores = recommend_talks(talk, names[0])["score"]
        print(talk[:40], "->", (scores == 0).sum())


Selected talk: Averting the climate crisis


,Exp 1: Title,Exp 2: Title + Tags,Exp 3: Title + Desc + Tags,Exp 4: Title + Desc + Transcript + Tags
Rank,,,,
1,New thinking on the climate crisis,New thinking on the climate crisis,New thinking on the climate crisis,New thinking on the climate crisis
2,The global learning crisis -- and what to do about it,The case for optimism on climate change,What comes after An Inconvenient Truth?,How we could change the planet's climate future
3,The state of the climate — and what we might do about it,The future of cars,The case for optimism on climate change,A climate solution where all sides can win
4,Why you should be a climate activist,Fusion is energy's future,The state of the climate — and what we might do about it,The disarming case to act right now on climate change
5,Why I must speak out about climate change,Why I must speak out about climate change,Why I must speak out about climate change,Climate change is happening. Here's how we adapt



Selected talk: Growing evidence of brain plasticity


,Exp 1: Title,Exp 2: Title + Tags,Exp 3: Title + Desc + Tags,Exp 4: Title + Desc + Transcript + Tags
Rank,,,,
1,Why can't we see evidence of alien life?,A brain in a supercomputer,Brain-to-brain communication has arrived. How we did it,How does income affect childhood brain development?
2,Growing new organs,What is consciousness?,3 clues to understanding your brain,A neural portrait of the human mind
3,The brain in love,How do you explain consciousness?,The left brain vs. right brain myth,"A monkey that controls a robot with its thoughts. No, really."
4,How to protect fast-growing cities from failing,Re-engineering the brain,A look inside the brain in real time,A look inside the brain in real time
5,Brain magic,3 clues to understanding your brain,How your brain's executive function works -- and how to improve it,What is so special about the human brain?



Selected talk: Do schools kill creativity?


,Exp 1: Title,Exp 2: Title + Tags,Exp 3: Title + Desc + Tags,Exp 4: Title + Desc + Transcript + Tags
Rank,,,,
1,4 lessons in creativity,Our failing schools. Enough is enough!,Bring on the learning revolution!,How to escape education's death valley
2,Our failing schools. Enough is enough!,Creative ways to get kids to thrive in school,How to escape education's death valley,Bring on the learning revolution!
3,Fashion and creativity,A father-daughter dance ... in prison,The child-driven education,Nerdcore comedy
4,Don't kill your language,Kids need structure,Let's raise kids to be entrepreneurs,A one-man world summit
5,Is space trying to kill us?,Use data to build better schools,How teachers can help kids find their political voices,Our failing schools. Enough is enough!


... (3 outputs left out)

,Exp 1: Title,Exp 2: Title + Tags,Exp 3: Title + Desc + Tags,Exp 4: Title + Desc + Transcript + Tags
Rank,,,,
1,What is love?,The brain in love,The brain in love,The brain in love
2,Love others to love yourself,What is love?,Technology hasn't changed love. Here's why,Technology hasn't changed love. Here's why
3,The brain in love,Love others to love yourself,What is love?,Falling in love is the easy part
4,How to find work you love,The science of sex,A better way to talk about love,A better way to talk about love
5,"Love, no matter what",Love -- you're doing it wrong,Falling in love is the easy part,Love others to love yourself



Overlap between experiments (out of 5):


,Talk,Exp1 vs Exp2,Exp1 vs Exp3,Exp1 vs Exp4,Exp3 vs Exp4
0,Averting the climate crisis,2,3,1,1
1,Growing evidence of brain plasticity,0,0,0,1
2,Do schools kill creativity?,1,0,1,2
3,Can we eat to starve cancer?,0,0,0,2
4,"Why we love, why we cheat",3,2,2,4



Experiment 1 recommendations with zero similarity:
Averting the climate crisis -> 0
Growing evidence of brain plasticity -> 0
Do schools kill creativity? -> 0
Can we eat to starve cancer? -> 0
Why we love, why we cheat -> 0



## 7. Evaluation and Comparison of Feature Combinations

The four experiments were compared using five selected TED Talks from different subject areas: climate change, brain science, education, cancer research, and love and relationships.

The recommendations were examined manually to assess how closely the suggested talks matched the subject of the selected talk. Recommendation overlap was also calculated to understand how much the results changed when different text features were included.

### Key Observations

1. **Title only:** This experiment sometimes produced loosely related recommendations because it relied mainly on words appearing in the talk titles.
2. **Title + Tags:** Adding topic tags improved subject matching, particularly for cancer-related and relationship-related talks.
3. **Title + Description + Tags:** This combination provided a good balance of topic information and produced relevant recommendations across the selected examples.
4. **Title + Description + Transcript + Tags:** Including transcripts introduced additional information, but did not consistently improve recommendation relevance compared with Experiment 3.

### Recommendation Overlap

The overlap analysis showed that the recommendations changed when additional features were included. Experiments 3 and 4 shared 10 out of 25 recommendations across the five selected talks. This indicates that transcript information can influence the recommendation results.

### Best Performing Feature Combination

Based on the manual inspection of the five selected examples, **Experiment 3 (Title + Description + Tags)** was selected as the preferred feature combination.

It provided a useful balance of title information, descriptive context, and topic tags without requiring the full transcript.

### Evaluation Limitation

This comparison was based on manual inspection of five selected TED Talks. It was not a formal evaluation using user ratings or a labelled relevance dataset. Therefore, the selected combination should be considered the most promising option among the four experiments tested, rather than a statistically proven best model.
